# Lab 1 – Vẽ hình cơ bản và thêm văn bản (OpenCV)

**Người thực hiện:** Sơn  
**Branch:** `feature/lab01-drawing-text`  
**Source:** `lab_1/src/drawing.py`

## Mục tiêu
Dùng OpenCV để vẽ **line**, **rectangle**, **circle** và thêm **text** lên ảnh; tạo một ảnh tổng hợp có đủ bốn loại đối tượng và lưu kết quả.

## 1. Import và đọc ảnh mẫu
Đường dẫn dùng đường dẫn tương đối (không dùng đường dẫn cá nhân).

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

sys.path.append(str(Path("..") / "src"))
import drawing as dr

from image_io import find_input_image
INPUT = find_input_image("06_animal_lion.jpg")   # ảnh trong data/input (tự chọn ảnh khác nếu không thấy)
OUTPUT_DIR = Path("../data/output")
FIG_DIR = Path("../result/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

image = dr.load_image(str(INPUT))          # báo lỗi nếu không đọc được ảnh
print("Shape:", image.shape, "(height, width, channels)")

# Tọa độ vẽ tính theo tỷ lệ kích thước ảnh để dùng được với mọi ảnh
H, W = image.shape[:2]
def px(fx, fy):
    """Đổi tỷ lệ (0-1) thành tọa độ pixel (x, y)."""
    return (int(fx * W), int(fy * H))
T = max(3, W // 300)     # độ dày nét
FS = max(0.8, W / 800)   # cỡ chữ

def show(img, title=""):
    """OpenCV lưu BGR, Matplotlib hiển thị RGB nên phải đổi trước khi vẽ."""
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")

plt.figure(figsize=(7, 4))
show(image, "Ảnh gốc")
plt.show()

## 2. Hệ tọa độ trong ảnh
- Gốc `(0, 0)` ở **góc trên-trái**.
- `x` tăng sang **phải**, `y` tăng xuống **dưới**.
- Điểm trong hàm vẽ viết theo dạng `(x, y)` (cột, hàng), còn khi truy cập mảng thì là `image[y, x]`.
- Màu trong OpenCV là **BGR**: `(255, 0, 0)` là xanh dương, `(0, 255, 0)` xanh lá, `(0, 0, 255)` đỏ.

## 3. Vẽ đường thẳng (line)
`cv2.line(img, pt1, pt2, color, thickness)`
- `pt1`, `pt2`: điểm đầu và điểm cuối `(x, y)`
- `color`: màu BGR
- `thickness`: độ dày nét (pixel)

In [ ]:
img_line = dr.draw_line(image, px(0.06, 0.10), px(0.40, 0.10), (255, 0, 0), T)
plt.figure(figsize=(7, 4)); show(img_line, f"Line (xanh dương, thickness={T})"); plt.show()

## 4. Vẽ hình chữ nhật (rectangle)
`cv2.rectangle(img, top_left, bottom_right, color, thickness)`
- Truyền **góc trên-trái** và **góc dưới-phải**.
- `thickness = -1` sẽ **tô đặc** hình.

In [ ]:
img_rect = dr.draw_rectangle(image, px(0.12, 0.20), px(0.40, 0.50), (0, 255, 0), T)
img_rect_fill = dr.draw_rectangle(image, px(0.12, 0.20), px(0.40, 0.50), (0, 255, 0), -1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); show(img_rect, f"Rectangle thickness={T}")
plt.subplot(1, 2, 2); show(img_rect_fill, "Rectangle thickness=-1 (tô đặc)")
plt.show()

## 5. Vẽ hình tròn (circle)
`cv2.circle(img, center, radius, color, thickness)`
- `center`: tâm `(x, y)`; `radius`: bán kính (pixel).

In [ ]:
img_circle = dr.draw_circle(image, px(0.65, 0.40), int(0.12 * W), (0, 0, 255), T)
plt.figure(figsize=(7, 4)); show(img_circle, "Circle (đỏ, r=80)"); plt.show()

## 6. Thêm văn bản (text)
`cv2.putText(img, text, org, fontFace, fontScale, color, thickness, lineType)`
- `org`: góc **dưới-trái** của dòng chữ.
- `fontFace`: kiểu font, ví dụ `cv2.FONT_HERSHEY_SIMPLEX`.
- `fontScale`: hệ số phóng chữ; `thickness`: độ dày nét chữ.
- `cv2.LINE_AA` làm mịn viền chữ.
- Font mặc định **không hỗ trợ tiếng Việt có dấu**, nên dùng text không dấu.

In [ ]:
img_text = dr.draw_text(image, "Computer Vision", px(0.06, 0.85),
                       cv2.FONT_HERSHEY_SIMPLEX, FS, (255, 255, 255), T)
plt.figure(figsize=(7, 4)); show(img_text, "Text"); plt.show()

## 7. Ảnh tổng hợp (line + rectangle + circle + text)

In [ ]:
combined = image.copy()
cv2.line(combined, px(0.06, 0.10), px(0.40, 0.10), (255, 0, 0), T)
cv2.rectangle(combined, px(0.12, 0.20), px(0.40, 0.50), (0, 255, 0), T)
cv2.circle(combined, px(0.65, 0.40), int(0.12 * W), (0, 0, 255), T)
cv2.putText(combined, "Computer Vision", px(0.06, 0.85),
            cv2.FONT_HERSHEY_SIMPLEX, FS, (255, 255, 255), T, cv2.LINE_AA)

plt.figure(figsize=(9, 5.5)); show(combined, "Ảnh tổng hợp"); plt.show()

## 8. Lưu kết quả

In [ ]:
ok = cv2.imwrite(str(OUTPUT_DIR / f"{INPUT.stem}_drawing.jpg"), combined)
print("Đã lưu ảnh:", ok, "->", OUTPUT_DIR / f"{INPUT.stem}_drawing.jpg")

# Lưu thêm hình minh họa so sánh trước/sau cho báo cáo
fig = plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1); show(image, "Trước")
plt.subplot(1, 2, 2); show(combined, "Sau khi vẽ")
fig.savefig(FIG_DIR / "drawing_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Giải thích tham số chính

| Hàm | Tham số quan trọng | Ý nghĩa |
|---|---|---|
| `cv2.line` | `pt1`, `pt2` | Hai đầu mút của đoạn thẳng, dạng `(x, y)` |
| `cv2.rectangle` | `top_left`, `bottom_right` | Hai góc đối diện; `thickness=-1` để tô đặc |
| `cv2.circle` | `center`, `radius` | Tâm `(x, y)` và bán kính tính bằng pixel |
| `cv2.putText` | `org`, `fontScale` | `org` là góc dưới-trái của chữ; `fontScale` quyết định cỡ chữ |
| tất cả | `color` | Truyền theo **BGR**, không phải RGB |
| tất cả | `thickness` | Độ dày nét theo pixel; số âm (-1) nghĩa là tô đặc (line/text không dùng) |

**Tọa độ:** gốc ở góc trên-trái, `x` sang phải, `y` xuống dưới.  
**Màu:** `(255, 0, 0)` hiển thị là xanh dương vì thứ tự là B-G-R.  
**Thickness:** nét càng dày thì hình càng nổi nhưng che nhiều nội dung ảnh hơn.

## 10. Nhận xét
- Các hàm vẽ của OpenCV **sửa trực tiếp lên mảng ảnh** (in-place). Vì vậy notebook dùng `image.copy()` để giữ ảnh gốc không bị thay đổi.
- Nếu truyền màu theo RGB thay vì BGR, màu hiển thị sẽ bị đảo (đỏ thành xanh dương). Cần nhớ OpenCV dùng BGR.
- Tọa độ viết cố định theo pixel nên khi đổi ảnh có kích thước khác, hình có thể lệch vị trí. Hàm `draw_all()` trong `drawing.py` tính tọa độ theo tỷ lệ kích thước ảnh để khắc phục điều này.
- `putText` không hiển thị được tiếng Việt có dấu; nếu cần, phải dùng Pillow (`ImageDraw.text` với font TTF).

## 11. Kiểm tra hàm tiện ích `draw_all()`
Hàm tự co tọa độ theo kích thước ảnh, dùng được với bất kỳ ảnh nào của nhóm.

In [ ]:
auto = dr.draw_all(image, "Computer Vision")
plt.figure(figsize=(9, 5.5)); show(auto, "draw_all() – tọa độ theo tỷ lệ"); plt.show()